In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
xy = torch.tensor([0.5, 1])
trajectory = []

parameters = [xy]
for p in parameters:
    p.requires_grad = True
opt = optim.SGD(parameters, lr=0.1, momentum=0)

def weierstrass_1d(t, a=0.5, b=3.0, n_terms=20):
    n = torch.arange(n_terms)
    return torch.sum((a ** n) * torch.cos((b ** n) * math.pi * t))

def f(x, y, a=0.5, b=3.0, n_terms=20):
    return weierstrass_1d(x, a, b, n_terms) + weierstrass_1d(y, a, b, n_terms)

def parabolic(x, y):
    return x**2+y**2

for i in range(10):
    x, y = xy[0], xy[1]
    trajectory.append((x.item(), y.item()))
    # z = x**2+y**2
    z = parabolic(x, y)
    print(f"f(<{x}, {y}>)={z}")
    z.backward()
    opt.step()
    opt.zero_grad()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

xs = np.linspace(-2, 2, 400)
ys = np.linspace(-2, 2, 400)
X, Y = np.meshgrid(xs, ys)

Z = parabolic(X, Y)

# plot
plt.figure(figsize=(7, 6))
plt.contourf(X, Y, Z, levels=100, cmap='viridis')

tra = np.array(trajectory)
plt.scatter(tra[:,0], tra[:, 1], c="r", s=0.5)
plt.xlim(-2, 2)
plt.ylim(-2, 2)

In [ ]:
x = torch.linspace(-2, 2, 1000)
plt.plot(x, list(map(weierstrass_1d, x)))

In [ ]:
import numpy as np
import torch
import trimesh
from torch.utils.data import TensorDataset, DataLoader

def stl_max_z_grid(stl_path: str, resolution: float):
    mesh = trimesh.load_mesh(stl_path)
    tri = np.asarray(mesh.triangles, dtype=np.float64)

    xyz_min = tri.reshape(-1, 3).min(axis=0)
    xyz_max = tri.reshape(-1, 3).max(axis=0)
    xmin, ymin, _ = xyz_min
    xmax, ymax, _ = xyz_max

    xs = np.arange(xmin, xmax + resolution, resolution)
    ys = np.arange(ymin, ymax + resolution, resolution)
    zmax = np.full((len(ys), len(xs)), np.nan, dtype=np.float64)

    for t in tri:
        (x1, y1, z1), (x2, y2, z2), (x3, y3, z3) = t

        den = (y2 - y3) * (x1 - x3) + (x3 - x2) * (y1 - y3)
        if abs(den) < 1e-15:
            continue

        txmin, tymin = t[:, :2].min(axis=0)
        txmax, tymax = t[:, :2].max(axis=0)

        ix0 = max(0, int(np.floor((txmin - xmin) / resolution)))
        ix1 = min(len(xs) - 1, int(np.ceil((txmax - xmin) / resolution)))
        iy0 = max(0, int(np.floor((tymin - ymin) / resolution)))
        iy1 = min(len(ys) - 1, int(np.ceil((tymax - ymin) / resolution)))
        if ix0 > ix1 or iy0 > iy1:
            continue

        XX, YY = np.meshgrid(xs[ix0:ix1 + 1], ys[iy0:iy1 + 1])

        w1 = ((y2 - y3) * (XX - x3) + (x3 - x2) * (YY - y3)) / den
        w2 = ((y3 - y1) * (XX - x3) + (x1 - x3) * (YY - y3)) / den
        w3 = 1.0 - w1 - w2

        inside = (
            (w1 >= -1e-12) & (w2 >= -1e-12) & (w3 >= -1e-12) &
            (w1 <= 1 + 1e-12) & (w2 <= 1 + 1e-12) & (w3 <= 1 + 1e-12)
        )
        if not inside.any():
            continue

        ZZ = w1 * z1 + w2 * z2 + w3 * z3
        patch = zmax[iy0:iy1 + 1, ix0:ix1 + 1]

        cur = patch[inside]
        new = ZZ[inside]
        patch[inside] = np.where(np.isnan(cur), new, np.maximum(cur, new))

    return xs, ys, zmax


def make_loader_from_stl(
    stl_path: str,
    resolution: float,
    xy_div: float = 10.0,
    batch_size: int = 1024,
    shuffle: bool = True,
    drop_last: bool = True,
):
    xs, ys, Z = stl_max_z_grid(stl_path, resolution)
    X, Y = np.meshgrid(xs, ys)

    finite = np.isfinite(Z)
    if finite.any():
        zmin = Z[finite].min()
        zmax = Z[finite].max()
        Z = (Z - zmin) / (zmax - zmin + 1e-12)

    Z = np.nan_to_num(Z, nan=1.0)
    plt.imshow(Z)
    plt.colorbar()

    x = torch.from_numpy(
        np.stack([X / xy_div, Y / xy_div], axis=-1).reshape(-1, 2)
    ).float()
    y = torch.from_numpy(Z.reshape(-1, 1)).float()

    ds = TensorDataset(x, y)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)
    return loader
# xs, ys, z = stl_max_z_grid("gwanak.stl", resolution=0.05)
# print(z.shape)     # (len(ys), len(xs))
# plt.figure(figsize=(8, 6))
# plt.imshow(
#     z,
#     origin="lower",
#     extent=[xs[0], xs[-1], ys[0], ys[-1]],
#     aspect="equal",
# )
# plt.colorbar(label="max z")
# plt.xlabel("x")
# plt.ylabel("y")
# plt.title("Maximum z over (x, y)")
# plt.show()
loader = make_loader_from_stl(
    "gwanak.stl",
    resolution=0.5,
    xy_div=10.0,
    batch_size=1024,
)

In [ ]:
d = 128
net = nn.Sequential(
    nn.Linear(2, d),
    nn.Tanh(),
    # nn.BatchNorm1d(d),
    nn.Linear(d, d),
    nn.Tanh(),
    # nn.BatchNorm1d(d),
    nn.Linear(d, d),
    nn.Tanh(),
    nn.Linear(d, d),
     nn.Tanh(),
    nn.Linear(d, d),

   nn.Tanh(),
    nn.Linear(d, d),


    # nn.BatchNorm1d(d),
    nn.Tanh(),
    nn.Linear(d, 1)
)

opt = optim.AdamW(net.parameters(), lr=0.001, weight_decay=1e-5)

In [ ]:
net.train()
i = 0
while i<30000:
    for x, y in loader:
        i += 1
        pred = net(x)
        loss = nn.functional.smooth_l1_loss(pred, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if (i+1)%100==0:
            print(loss.item())

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

xy = nn.Parameter(torch.tensor([[-0.6, 1]], dtype=torch.float32))
trajectory = []

# walk_opt = optim.Adam([xy], lr=0.27, betas=(0.95, 0.998)) for 1, -1
walk_opt = optim.Adam([xy], lr=0.074, betas=(0.95, 0.998))

net.eval()
for p in net.parameters():
    p.requires_grad_(False)
for i in range(10000):
    trajectory.append((xy[0,0].item(), xy[0,1].item()))

    walk_opt.zero_grad()
    z = net(xy).squeeze()
    z.backward()
    walk_opt.step()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
bmin, bmax = -4, 4
XX, YY = torch.meshgrid(torch.linspace(bmin, bmax, 400), torch.linspace(bmin, bmax, 400))
shape = XX.shape
xy = torch.stack([XX, YY], dim=2).view(-1, 2)

net.eval()
ZZ = net(xy).view(*shape)

# plot
plt.figure(figsize=(7, 6))
plt.contourf(XX.detach(), YY.detach(), ZZ.detach(), levels=100, cmap='viridis')

tra = np.array(trajectory)
plt.scatter(tra[:,0], tra[:, 1], c="r", s=1)
plt.colorbar()
plt.xlim(bmin, bmax)
plt.ylim(bmin, bmax)